# 🎯 inzva DLSG#10 — CIFAR-100 Classification
## Complete Educational Pipeline: From Scratch CNN to ResNet18 Transfer Learning

**Competition Goal:** Classify 10,000 test images into 100 CIFAR-100 classes.

**Milestones:**
1. 🧱 Baseline CNN from Scratch
2. 🛡️ Data Augmentation & Regularization
3. ⚙️ Training Dynamics (Hyperparameter Tuning)
4. 🚀 Transfer Learning with ResNet18
5. 🔍 Interpretability with Captum
6. 🐸 OOD Testing

---
Look for **📝 LEARN:** comments throughout the code — they explain the *why* behind every technique!

## 0️⃣ Setup — GPU Check & Dependencies

⚠️ **IMPORTANT**: After running this cell, if it says "Restart Required", go to **Runtime → Restart session**, then **re-run this cell** and continue.

In [ ]:
# ============================================================================
# 📝 FIX: Install captum without breaking NumPy/torchvision compatibility
# --no-deps prevents captum from downgrading numpy
# ============================================================================
import subprocess, sys

try:
    import captum
    print('✅ Captum already installed')
except ImportError:
    print('📦 Installing captum (without changing numpy)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'captum', '--no-deps', '-q'])
    print('✅ Captum installed! If you see any errors below, do:')
    print('   Runtime → Restart session, then re-run this cell.')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
import os
import time
from PIL import Image

# Check captum imports
try:
    from captum.attr import Occlusion
    CAPTUM_AVAILABLE = True
    print('✅ Captum loaded successfully')
except ImportError:
    CAPTUM_AVAILABLE = False
    print('⚠️ Captum not available — Milestones 5 & 6 captum features will be skipped')

# 📝 LEARN: Check if GPU is available
# Colab provides free GPU access (usually Tesla T4 with 15GB VRAM)
# GPU makes training 10-50x faster than CPU!
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

print(f'\n📦 Versions: torch={torch.__version__}, numpy={np.__version__}')

## 📦 Data Loading Utilities

📝 **LEARN: Why do we normalize images?**
- Raw pixel values are 0-255, which creates very large gradients
- Neural networks train better with small values centered around 0
- Two strategies:
  - **CIFAR-100 stats**: Mean/Std from CIFAR-100 training set (for custom CNNs)
  - **ImageNet stats**: Must use when loading pre-trained models!

In [ ]:
# ============================================================================
# 📝 LEARN: Normalization Constants
# ============================================================================
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# CIFAR-100 class names (100 fine-grained classes)
CIFAR100_CLASSES = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver',
    'bed', 'bee', 'beetle', 'bicycle', 'bottle',
    'bowl', 'boy', 'bridge', 'bus', 'butterfly',
    'camel', 'can', 'castle', 'caterpillar', 'cattle',
    'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach',
    'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox',
    'girl', 'hamster', 'house', 'kangaroo', 'keyboard',
    'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard',
    'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain',
    'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid',
    'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree',
    'plain', 'plate', 'poppy', 'porcupine', 'possum',
    'rabbit', 'raccoon', 'ray', 'road', 'rocket',
    'rose', 'sea', 'seal', 'shark', 'shrew',
    'skunk', 'skyscraper', 'snail', 'snake', 'spider',
    'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
    'tank', 'telephone', 'television', 'tiger', 'tractor',
    'train', 'trout', 'tulip', 'turtle', 'wardrobe',
    'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]


def get_cifar100_transforms(augment=False, image_size=32, use_imagenet_stats=False):
    """
    📝 LEARN: Transforms are applied to each image before feeding to the model.
    - ToTensor: converts pixels from 0-255 to 0.0-1.0
    - Normalize: centers values around 0
    - Augmentation transforms: only during training!
    """
    mean = IMAGENET_MEAN if use_imagenet_stats else CIFAR100_MEAN
    std = IMAGENET_STD if use_imagenet_stats else CIFAR100_STD

    if augment:
        # 📝 LEARN: Each time we load an image, RANDOM transforms are applied
        # This means the model sees different versions each epoch!
        transform_list = [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomCrop(image_size, padding=4),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    else:
        transform_list = [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]

    return transforms.Compose(transform_list)


def get_cifar100_dataloaders(batch_size=64, augment=False, image_size=32,
                              use_imagenet_stats=False, val_split=0.1, num_workers=2):
    """
    📝 LEARN: DataLoaders feed batches of images to the model.
    We split training data into train (90%) and validation (10%)
    to monitor overfitting.
    """
    train_transform = get_cifar100_transforms(augment=augment, image_size=image_size,
                                              use_imagenet_stats=use_imagenet_stats)
    val_transform = get_cifar100_transforms(augment=False, image_size=image_size,
                                             use_imagenet_stats=use_imagenet_stats)

    full_train = datasets.CIFAR100(root='./data', train=True, download=True,
                                   transform=train_transform)
    val_data = datasets.CIFAR100(root='./data', train=True, download=True,
                                 transform=val_transform)

    # Fixed seed for reproducible split
    num_train = len(full_train)
    indices = list(range(num_train))
    np.random.seed(42)
    np.random.shuffle(indices)
    split = int(np.floor(val_split * num_train))

    train_loader = DataLoader(Subset(full_train, indices[split:]),
                              batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(Subset(val_data, indices[:split]),
                            batch_size=batch_size, shuffle=False, num_workers=num_workers)

    print(f'📊 Train: {num_train - split} | Val: {split} | Batch: {batch_size} | '
          f'Size: {image_size}x{image_size} | Aug: {augment} | '
          f'Norm: {"ImageNet" if use_imagenet_stats else "CIFAR"}')
    return train_loader, val_loader

print('✅ Data utilities loaded!')

## 🏋️ Training Utilities

📝 **LEARN: The Training Loop — The Heart of Deep Learning**
1. **FORWARD**: Input → Model → Prediction
2. **LOSS**: Compare prediction with true label
3. **BACKWARD**: Compute gradients (backpropagation)
4. **UPDATE**: Optimizer adjusts weights
5. **REPEAT** for every batch, for many epochs

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """📝 LEARN: One epoch = one complete pass through all training data."""
    model.train()  # Enable dropout, batchnorm training mode
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, desc='  Train', leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)         # Forward pass
        loss = criterion(outputs, labels) # Compute loss

        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute new gradients (backprop!)
        optimizer.step()       # Update weights

        total_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """📝 LEARN: Evaluation — no gradient computation, no weight updates."""
    model.eval()  # Disable dropout, use running stats for batchnorm
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, desc='  Val', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()

    return total_loss / total, 100.0 * correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=10, scheduler=None, save_name='best_model.pth'):
    """Full training pipeline with best model saving."""
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0

    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n🚀 Training {num_epochs} epochs | Params: {params:,} ({trainable:,} trainable) | Device: {device}')
    print('=' * 80)

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        saved = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_name)
            saved = ' ⭐ SAVED'

        lr = optimizer.param_groups[0]['lr']
        gap = train_acc - val_acc
        gap_warn = f' ⚠️ GAP:{gap:.0f}%' if gap > 20 else ''
        print(f'  [{epoch+1}/{num_epochs}] {time.time()-t0:.0f}s | '
              f'Train: {train_loss:.4f}/{train_acc:.1f}% | '
              f'Val: {val_loss:.4f}/{val_acc:.1f}% | '
              f'LR: {lr:.6f}{saved}{gap_warn}')

    print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.2f}%')
    return history


def plot_history(history, title='Training'):
    """📝 LEARN: Plotting helps diagnose underfitting vs overfitting."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train', color='#FF6B6B', lw=2)
    ax1.plot(history['val_loss'], label='Val', color='#4ECDC4', lw=2)
    ax1.set_title(f'{title} — Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history['train_acc'], label='Train', color='#FF6B6B', lw=2)
    ax2.plot(history['val_acc'], label='Val', color='#4ECDC4', lw=2)
    ax2.set_title(f'{title} — Accuracy (%)'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print('✅ Training utilities loaded!')

---
# 🧱 Milestone 1: Baseline CNN from Scratch

📝 **LEARN: What is a CNN?**
- **Conv2d**: A small filter slides across the image, detecting patterns (edges, textures)
- **MaxPool2d**: Shrinks the image by keeping strongest activations
- **ReLU**: Non-linearity → `max(0, x)`. Without it, stacking layers = one big linear layer
- **Linear**: Maps extracted features → class scores

**Goal**: Beat random guessing (1% = 1/100 classes)

In [ ]:
class SimpleCNN(nn.Module):
    """
    📝 LEARN: 3 conv blocks → classifier

    Receptive Field (RF): area of original image a deep neuron "sees"
    - After 1 conv (3x3): RF = 3x3
    - After 2 conv (3x3): RF = 5x5
    - After pooling: RF doubles!
    More layers → larger RF → sees bigger patterns
    """
    def __init__(self, num_classes=100):
        super().__init__()
        # Block 1: 3→32 channels | 32x32 → 16x16
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        # Block 2: 32→64 channels | 16x16 → 8x8
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        # Block 3: 64→128 channels | 8x8 → 4x4
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        # Classifier: 128*4*4=2048 → 256 → 100
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256),
            nn.ReLU(True), nn.Linear(256, num_classes))

    def forward(self, x):
        # 📝 Data flows: edges → textures → object parts → class scores
        return self.classifier(self.block3(self.block2(self.block1(x))))


# ── Train Baseline CNN ─────────────────────────────────
print('🧱 Milestone 1: Baseline CNN')
train_loader, val_loader = get_cifar100_dataloaders(batch_size=128, augment=False, image_size=32)

model_m1 = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_m1.parameters(), lr=1e-3)

history_m1 = train_model(model_m1, train_loader, val_loader, criterion, optimizer,
                          num_epochs=15, save_name='m1_baseline.pth')
plot_history(history_m1, 'Milestone 1: Baseline CNN')

print(f'\n✅ Beat random guessing? {"YES!" if max(history_m1["val_acc"]) > 1 else "No"}')
print(f'   Best val accuracy: {max(history_m1["val_acc"]):.2f}%')

---
# 🛡️ Milestone 2: Data Augmentation & Regularization

📝 **LEARN: The Overfitting Problem**
- 500 images/class → model can MEMORIZE them
- Train acc: 95%, Val acc: 20% → it learned the images, not the concepts!

**Two weapons:**
1. **Augmentation**: Random flips, crops, color changes → "expands" dataset
2. **Dropout**: Randomly turns off neurons → forces redundant learning

In [ ]:
class AugmentedCNN(nn.Module):
    """📝 Same arch as M1, but with Dropout after each block."""
    def __init__(self, num_classes=100):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.1))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.3))
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((2, 2)))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256*2*2, 512),
            nn.ReLU(True), nn.Dropout(0.5), nn.Linear(512, num_classes))

    def forward(self, x):
        return self.classifier(self.block4(self.block3(self.block2(self.block1(x)))))


# ── Train with Augmentation ───────────────────────────
print('🛡️ Milestone 2: Augmentation + Dropout')
train_loader, val_loader = get_cifar100_dataloaders(batch_size=128, augment=True, image_size=32)

model_m2 = AugmentedCNN().to(device)
optimizer = optim.Adam(model_m2.parameters(), lr=1e-3)

history_m2 = train_model(model_m2, train_loader, val_loader, criterion, optimizer,
                          num_epochs=20, save_name='m2_augmented.pth')
plot_history(history_m2, 'Milestone 2: Augmentation + Dropout')

print(f'\n📊 Compare with M1:')
print(f'   M1 best val: {max(history_m1["val_acc"]):.2f}%')
print(f'   M2 best val: {max(history_m2["val_acc"]):.2f}%')
print(f'   Improvement: {max(history_m2["val_acc"]) - max(history_m1["val_acc"]):+.2f}%')

---
# ⚙️ Milestone 3: Training Dynamics

📝 **LEARN: Hyperparameters — the knobs YOU turn**
- **Learning Rate**: How big are the weight updates?
- **SGD vs Adam**: Different optimization strategies
- **Scheduler**: Automatically reduce LR when progress stalls
- **Weight Decay**: Penalize large weights (L2 regularization)

In [ ]:
# ── Experiment: AdamW + ReduceLROnPlateau + Weight Decay ──
# 📝 LEARN: This combo is often the best for quick experiments:
#   - AdamW: Adam with properly decoupled weight decay
#   - ReduceLROnPlateau: Reduce LR when val_loss stops improving
#   - Weight Decay: Penalizes large weights → smoother decision boundaries

print('⚙️ Milestone 3: AdamW + Scheduler + Weight Decay')
train_loader, val_loader = get_cifar100_dataloaders(batch_size=128, augment=True, image_size=32)

model_m3 = AugmentedCNN().to(device)
optimizer = optim.AdamW(model_m3.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history_m3 = train_model(model_m3, train_loader, val_loader, criterion, optimizer,
                          num_epochs=25, scheduler=scheduler, save_name='m3_tuned.pth')
plot_history(history_m3, 'Milestone 3: AdamW + Scheduler')

print(f'\n📊 Progress so far:')
print(f'   M1 (baseline):     {max(history_m1["val_acc"]):.2f}%')
print(f'   M2 (augmentation): {max(history_m2["val_acc"]):.2f}%')
print(f'   M3 (tuned):        {max(history_m3["val_acc"]):.2f}%')

---
# 🚀 Milestone 4: Transfer Learning with ResNet18

📝 **LEARN: Standing on the Shoulders of Giants**
- ResNet18 was trained on **1.2 million** ImageNet images
- It already knows about edges, textures, shapes, objects
- We just need to teach it OUR 100 classes

📝 **LEARN: ResNet's Secret — Skip Connections**
- Deeper networks should be better, but they got WORSE (degradation)
- ResNet fix: `output = F(x) + x` (shortcut/skip connection)
- If a layer isn't helpful, it learns `F(x) = 0`, effectively skipping itself

⚠️ **CRITICAL**: Pre-trained models MUST use ImageNet normalization!

In [ ]:
print('🚀 Milestone 4: Transfer Learning with ResNet18')

# ⚠️ CRITICAL: ImageNet normalization + 224x224 for pre-trained models!
train_loader, val_loader = get_cifar100_dataloaders(
    batch_size=64, augment=True, image_size=224, use_imagenet_stats=True)

# Load pre-trained ResNet18
model_m4 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 📝 LEARN: Replace final layer (1000 ImageNet classes → 100 CIFAR classes)
num_features = model_m4.fc.in_features  # 512
model_m4.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 100)
)
model_m4 = model_m4.to(device)

# 📝 LEARN: Differential Learning Rates
# Backbone (pre-trained): LOW LR — small adjustments only
# Head (new layer): HIGHER LR — needs to learn from scratch
backbone = [p for n, p in model_m4.named_parameters() if 'fc' not in n]
head = [p for n, p in model_m4.named_parameters() if 'fc' in n]

optimizer = optim.AdamW([
    {'params': backbone, 'lr': 1e-4},   # Low LR for pre-trained layers
    {'params': head, 'lr': 1e-3}         # Higher LR for new classifier
], weight_decay=1e-2)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history_m4 = train_model(model_m4, train_loader, val_loader, criterion, optimizer,
                          num_epochs=20, scheduler=scheduler, save_name='m4_resnet18.pth')
plot_history(history_m4, 'Milestone 4: ResNet18 Transfer Learning')

print(f'\n📊 All milestones:')
for name, h in [('M1 baseline', history_m1), ('M2 augmentation', history_m2),
                ('M3 tuned', history_m3), ('M4 ResNet18', history_m4)]:
    print(f'   {name:20s}: {max(h["val_acc"]):.2f}%')

---
# 🔍 Milestone 5: Interpretability with Captum

📝 **LEARN: Peek Inside the Black Box**
- **Occlusion**: Slide a grey patch across the image, measure prediction change
- If covering a region DROPS confidence → that region is IMPORTANT
- Helps understand: Is the model looking at the object or the background?

In [ ]:
if not CAPTUM_AVAILABLE:
    print('⚠️ Captum not available. Please restart runtime and re-run the Setup cell.')
else:
    from captum.attr import Occlusion

    def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
        tensor = tensor.clone()
        for t, m, s in zip(tensor, mean, std):
            t.mul_(s).add_(m)
        return tensor.clamp_(0, 1)

    # Load best model
    model_m4.load_state_dict(torch.load('m4_resnet18.pth', map_location=device))
    model_m4.eval()

    # Get sample images from CIFAR-100 test set
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    test_data = datasets.CIFAR100('./data', train=False, download=True, transform=test_transform)
    np.random.seed(42)
    sample_indices = np.random.choice(len(test_data), 4, replace=False)

    occlusion = Occlusion(model_m4)
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))

    for i, idx in enumerate(sample_indices):
        img, true_label = test_data[idx]
        inp = img.unsqueeze(0).to(device)

        with torch.no_grad():
            out = model_m4(inp)
            pred = out.argmax(1).item()
            conf = torch.softmax(out, 1)[0, pred].item()

        # Occlusion attribution
        attr = occlusion.attribute(inp, target=pred,
            sliding_window_shapes=(3, 16, 16), strides=(3, 8, 8), baselines=0)
        attr_map = attr.squeeze().cpu().numpy()
        attr_map = np.mean(np.abs(attr_map), axis=0)

        # Display
        orig = denormalize(img).permute(1, 2, 0).numpy()
        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f'True: {CIFAR100_CLASSES[true_label]}'); axes[i, 0].axis('off')
        axes[i, 1].imshow(attr_map, cmap='hot')
        axes[i, 1].set_title(f'Attribution ({CIFAR100_CLASSES[pred]} {conf:.0%})'); axes[i, 1].axis('off')
        axes[i, 2].imshow(orig); axes[i, 2].imshow(attr_map, cmap='hot', alpha=0.5)
        correct = '✅' if pred == true_label else '❌'
        axes[i, 2].set_title(f'Overlay {correct}'); axes[i, 2].axis('off')

    plt.suptitle('Milestone 5: Occlusion Attribution', fontsize=16)
    plt.tight_layout(); plt.show()

---
# 🐸 Milestone 6: OOD Frog Testing

Upload `frog.png` and `frog-flip.png` to Colab, then run the cell below.

📝 **LEARN: Invariance Check** — Does flipping change the prediction?
If yes, the model relies on spatial positioning, not semantic features.

In [ ]:
# Upload frog images to Colab first!
# You can drag-and-drop them in the Files panel (left sidebar)

from google.colab import files

frog_files = ['frog.png', 'frog-flip.png']
missing = [f for f in frog_files if not os.path.exists(f)]
if missing:
    print(f'📤 Please upload: {missing}')
    uploaded = files.upload()

# Define test_transform if not already defined (in case M5 was skipped)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

model_m4.eval()
fig, axes = plt.subplots(len(frog_files), 3, figsize=(15, 5 * len(frog_files)))
if len(frog_files) == 1: axes = axes.reshape(1, -1)

predictions = []
for i, fname in enumerate(frog_files):
    if not os.path.exists(fname):
        print(f'⚠️ {fname} not found, skipping'); continue

    img = Image.open(fname).convert('RGB')
    tensor = test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model_m4(tensor)
        probs = torch.softmax(out, 1)[0]
        top5_p, top5_i = probs.topk(5)

    pred = top5_i[0].item()
    predictions.append((fname, pred, top5_p[0].item()))

    print(f'\n📷 {fname}:')
    for j in range(5):
        print(f'   {"👑" if j==0 else "  "} {CIFAR100_CLASSES[top5_i[j]]:20s} {top5_p[j]:.1%}')

    axes[i, 0].imshow(img); axes[i, 0].set_title(fname); axes[i, 0].axis('off')
    axes[i, 1].barh([CIFAR100_CLASSES[idx] for idx in top5_i.cpu()][::-1],
                     top5_p.cpu().numpy()[::-1])
    axes[i, 1].set_title('Top-5'); axes[i, 1].set_xlim(0, 1)

    # Captum analysis (if available)
    if CAPTUM_AVAILABLE:
        from captum.attr import Occlusion
        occlusion = Occlusion(model_m4)
        attr = occlusion.attribute(tensor, target=pred,
            sliding_window_shapes=(3, 16, 16), strides=(3, 8, 8), baselines=0)
        attr_map = np.mean(np.abs(attr.squeeze().cpu().numpy()), axis=0)
        def denorm(t):
            t = t.clone()
            for c, m, s in zip(t, IMAGENET_MEAN, IMAGENET_STD): c.mul_(s).add_(m)
            return t.clamp_(0, 1)
        disp = denorm(tensor.squeeze().cpu()).permute(1, 2, 0).numpy()
        axes[i, 2].imshow(disp); axes[i, 2].imshow(attr_map, cmap='hot', alpha=0.5)
        axes[i, 2].set_title(f'Captum: {CIFAR100_CLASSES[pred]}'); axes[i, 2].axis('off')
    else:
        axes[i, 2].text(0.5, 0.5, 'Captum N/A', ha='center', va='center', fontsize=14)
        axes[i, 2].axis('off')

plt.tight_layout(); plt.show()

# Invariance check
if len(predictions) >= 2:
    print(f'\n🔄 Invariance: {"✅ Same prediction" if predictions[0][1] == predictions[1][1] else "⚠️ Different predictions — spatial bias detected!"}')

---
# 📤 Generate Kaggle Submission

Upload your `test_images/` folder to Colab, then run below.

**Steps:**
1. Upload test_images to Colab (or mount Google Drive)
2. Run the cell to generate `submission.csv`
3. Download and submit to Kaggle!

In [ ]:
# ── Option A: Upload test_images as a zip ──────────────
# Uncomment if you uploaded test_images.zip:
# !unzip -q test_images.zip -d test_images

# ── Option B: Mount Google Drive ──────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# TEST_DIR = '/content/drive/MyDrive/path/to/test_images'

TEST_DIR = 'test_images'  # Change this to your test images path

if not os.path.exists(TEST_DIR):
    print(f'❌ {TEST_DIR} not found!')
    print('   Upload test_images folder or mount Google Drive')
else:
    # Custom dataset for test images
    class TestDataset(Dataset):
        def __init__(self, test_dir, transform):
            self.test_dir = test_dir
            self.transform = transform
            self.files = sorted([f for f in os.listdir(test_dir) if f.endswith('.png')])
        def __len__(self): return len(self.files)
        def __getitem__(self, idx):
            img = Image.open(os.path.join(self.test_dir, self.files[idx])).convert('RGB')
            return self.transform(img), self.files[idx]

    # Define test_transform if not already defined
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    test_dataset = TestDataset(TEST_DIR, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
    print(f'🧪 Test images: {len(test_dataset)}')

    # Load best model and predict
    model_m4.load_state_dict(torch.load('m4_resnet18.pth', map_location=device))
    model_m4.eval()

    all_names, all_preds = [], []
    with torch.no_grad():
        for images, names in tqdm(test_loader, desc='Predicting'):
            outputs = model_m4(images.to(device))
            _, preds = torch.max(outputs, 1)
            all_names.extend(names)
            all_preds.extend(preds.cpu().numpy())

    submission = pd.DataFrame({'filename': all_names, 'class': all_preds})
    submission.to_csv('submission.csv', index=False)

    print(f'\n✅ submission.csv saved! ({len(submission)} predictions)')
    print(submission.head())

    # Download
    from google.colab import files
    files.download('submission.csv')

---
# 🎉 Congratulations!

You've completed all 6 milestones! Here's what you learned:

| Milestone | Concept | Key Takeaway |
|---|---|---|
| 1 | CNN from scratch | Conv → Pool → Linear pipeline |
| 2 | Augmentation | Fighting overfitting with data diversity |
| 3 | Hyperparameters | LR, optimizer, scheduler, weight decay |
| 4 | Transfer Learning | Pre-trained models are powerful! |
| 5 | Interpretability | Understanding what models "see" |
| 6 | OOD Testing | Real-world robustness |

🚀 **Submit `submission.csv` to Kaggle and share your score!**